# 03. Entity Resolution Profile Mart: multi-value витрина профилей


## Префиксы признаков

В витрине и правилах blocking признаки записываются с коротким префиксом источника. Это нужно, чтобы по названию сразу было видно, откуда пришло значение и как его можно использовать.

- `identity` — идентификационные поля профиля: телефон, email, имя и похожие признаки. В обучении и проверках они полезны, но в blocking используем их осторожно: телефон/email могут быть сильным сигналом, а имя само по себе слишком шумное.
- `np` — признаки из `non_processing_features`: контекст устройства и окружения, например гео, устройство, браузер, операционная система.
- `rt` — признаки из `realtime_features`: контекст события в момент регистрации или активности, например гео/время из realtime-слоя.
- `fs` — признаки из `fs_features`: поведенческие и продуктовые признаки, например посещения, клики, аккаунты, заказы, site-id и postman-события.
- `derived` — производные признаки, которые мы рассчитываем внутри пайплайна из исходных данных. Например, временная корзина регистрации: ночь/утро/рабочее время/вечер или окно регистрации +- 60 минут.


In [1]:
from pathlib import Path
import gc
import json
import hashlib
import re
import unicodedata
from scipy.stats import binomtest

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

## 1. Пути и настройки

На выходе этого ноутбука собираем только слои, которые нужны дальше для blocking, обучения и inference.


In [2]:
RAW_DATA_DIR = Path("../data")
PROCESSED_DATA_DIR = Path("../data/processed")

INPUT_PATH = RAW_DATA_DIR / "split_label_train_V3.snappy.parquet"
OUTPUT_DIR = PROCESSED_DATA_DIR / "er_profile_mart_multivalue"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_COLS = ['event_id', 'profile_id', 'entity_id', 'created_at']

INPUT_PATH, OUTPUT_DIR


(WindowsPath('../data/split_label_train_V3.snappy.parquet'),
 WindowsPath('../data/processed/er_profile_mart_multivalue'))

## 2. Загрузка исходных событий

Исходная гранулярность: одна строка = одно событие / наблюдение профиля. Дальше агрегируем до уровня `profile_id`, но не теряем наборы значений.


In [3]:
raw = pd.read_parquet(INPUT_PATH).reset_index(drop=True)
# event_id позволяет понять, из какого конкретно события пришло
# значение;по нему считается value_count=('event_id', 'size');
# он помогает не потерять связь между raw-событием и
# распарсенными identity / np / rt / fs значениями;
raw['event_id'] = raw.index.astype('int64')
raw['created_at'] = pd.to_datetime(raw['created_at'], errors='coerce')

print('raw shape:', raw.shape)
print('n profiles:', raw['profile_id'].nunique())
print('n entities:', raw['entity_id'].nunique())
raw.head(1)

raw shape: (68036, 13)
n profiles: 61927
n entities: 53369


,created_at,first_name,last_name,email,phone,birthday,sex,non_processing_features,realtime_features,fs_features,profile_id,entity_id,event_id
0,2025-11-01 00:27:04.995,Анфиса,None,lqvxvltxx@mail.ru,None,None,female,"[device:smartphone, geoname_id:2013348, browse...","{""country"":""RU"",""is_million"":false,""tz_offset""...","[visited_30:250, visited_365:1813, visited_30:...",b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,0


## 3. Нормализация и парсинг

Разделяем:

- `value_raw_text` — исходное значение в parquet-safe текстовом виде;
- `value_norm` — нормализованное значение для анализа и blocking;
- `source` — семейство признаков: `identity`, `np`, `rt`, `fs`, `derived`;
- `feature_key` — стабильное имя вида `source__feature`.


In [4]:
def norm_text(x):
    if pd.isna(x):
        return pd.NA
    s = unicodedata.normalize('NFKC', str(x)).strip().lower()
    s = re.sub(r'\s+', ' ', s)
    return s if s else pd.NA


def norm_phone(x):
    if pd.isna(x):
        return pd.NA
    digits = re.sub(r'\D+', '', str(x))
    # Простая нормализация для РФ: 8XXXXXXXXXX -> 7XXXXXXXXXX.
    if len(digits) == 11 and digits.startswith('8'):
        digits = '7' + digits[1:]
    return digits if digits else pd.NA


def norm_date_ymd(x):
    dt = pd.to_datetime(x, errors='coerce')
    if pd.isna(dt):
        return pd.NA
    return dt.strftime('%Y-%m-%d')


def normalize_identity_value(feature: str, value):
    if feature == 'phone':
        return norm_phone(value)
    if feature == 'birthday':
        return norm_date_ymd(value)
    return norm_text(value)


def parse_kv_token(token):
    # 'browser:chrome' -> ('browser', 'chrome')
    # 'is_phone'       -> ('is_phone', True)
    if token is None:
        return None, None
    s = str(token).strip()
    if not s:
        return None, None
    if ':' in s:
        key, value = s.split(':', 1)
        return key.strip(), value.strip()
    return s.strip(), True


def norm_feature_value(value):
    if value is True:
        return 'true'
    if value is False:
        return 'false'
    return norm_text(value)


def value_to_text(value):
    # Нужен единый тип для parquet: object-колонка с date/int/bool/string может падать при записи.
    if value is None or pd.isna(value):
        return pd.NA
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False, default=str)
    return str(value)

## 4. `profile_core`: базовая таблица профилей

Одна строка = один `profile_id`.  
`entity_id`, `entity_size`, `entity_kind` — это ground truth / label-derived metadata. Их можно использовать для оценки blocking и разметки пар, но нельзя использовать как production-признаки.


In [5]:
# profile_core отвечает на вопрос: “какие профили вообще есть и какая у них служебная информация”.
profile_core = (
    raw.groupby('profile_id', dropna=False, sort=False)
       .agg(
           event_count=('event_id', 'size'),
           entity_id=('entity_id', 'first'),
           entity_id_nunique=('entity_id', 'nunique'),
           first_event_at=('created_at', 'min'),
           last_event_at=('created_at', 'max'),
       )
       .reset_index()
)

entity_profile_counts = (
    raw.groupby('entity_id', sort=False)['profile_id']
       .nunique()
       .rename('entity_size')
       .reset_index()
)

profile_core = profile_core.merge(entity_profile_counts, on='entity_id', how='left')
profile_core['entity_kind'] = np.where(
    profile_core['entity_size'].fillna(0).astype(int) > 1,
    'multi_profile_entity',
    'single_profile_entity',
)

profile_core.to_parquet(OUTPUT_DIR / 'profile_core.parquet', index=False)
print('profile_core shape:', profile_core.shape)


profile_core shape: (61927, 8)


## 5. Временный event-value слой

На этом шаге собираем промежуточную таблицу значений на уровне события.


In [6]:
# 5.1 Identity-признаки
identity_parts = []

IDENTITY_COLS = ['first_name', 'last_name', 'email', 'phone', 'birthday', 'sex']

for col in IDENTITY_COLS:
    tmp = raw[BASE_COLS + [col]].rename(columns={col: 'value_raw'}).copy()
    tmp = tmp[tmp['value_raw'].notna()]
    if tmp.empty:
        continue

    tmp['source'] = 'identity'
    tmp['feature'] = col
    tmp['value_norm'] = tmp['value_raw'].map(lambda x, c=col: normalize_identity_value(c, x))
    tmp = tmp[tmp['value_norm'].notna()]
    identity_parts.append(tmp[BASE_COLS + ['source', 'feature', 'value_raw', 'value_norm']])

identity_event_long = (
    pd.concat(identity_parts, ignore_index=True)
    if identity_parts
    else pd.DataFrame(columns=BASE_COLS + ['source', 'feature', 'value_raw', 'value_norm'])
)

del identity_parts
gc.collect()

print('identity_event_long shape:', identity_event_long.shape)


identity_event_long shape: (161493, 8)


In [7]:
# 5.2 non_processing_features и fs_features: списки token'ов вида key:value или flag
rows = []

for source_col, source_name in [('non_processing_features', 'np'), ('fs_features', 'fs')]:
    arr = raw[BASE_COLS + [source_col]].to_numpy(dtype=object)

    for event_id, profile_id, entity_id, created_at, values in arr:
        if values is None:
            continue

        for token in list(values):
            feature, value = parse_kv_token(token)
            if feature is None:
                continue

            value_norm = norm_feature_value(value)
            if pd.isna(value_norm):
                continue

            rows.append((event_id, profile_id, entity_id, created_at, source_name, feature, value, value_norm))

list_event_long = pd.DataFrame.from_records(
    rows,
    columns=BASE_COLS + ['source', 'feature', 'value_raw', 'value_norm'],
)

del rows
gc.collect()

print('list_event_long shape:', list_event_long.shape)


list_event_long shape: (1499043, 8)


In [8]:
# 5.3 realtime_features: JSON-объект
rows = []

arr = raw[BASE_COLS + ['realtime_features']].to_numpy(dtype=object)

for event_id, profile_id, entity_id, created_at, payload in arr:
    if payload is None or pd.isna(payload):
        continue

    try:
        obj = json.loads(payload)
    except Exception:
        continue

    if not isinstance(obj, dict):
        continue

    for feature, value in obj.items():
        value_norm = norm_feature_value(value)
        if pd.notna(value_norm):
            rows.append((event_id, profile_id, entity_id, created_at, 'rt', str(feature), value, value_norm))

realtime_event_long = pd.DataFrame.from_records(
    rows,
    columns=BASE_COLS + ['source', 'feature', 'value_raw', 'value_norm'],
)

del rows
gc.collect()

print('realtime_event_long shape:', realtime_event_long.shape)


realtime_event_long shape: (450144, 8)


In [9]:
profile_event_value_long = pd.concat(
    [identity_event_long, list_event_long, realtime_event_long],
    ignore_index=True,
)

del identity_event_long, list_event_long, realtime_event_long
gc.collect()

profile_event_value_long['value_raw_text'] = profile_event_value_long['value_raw'].map(value_to_text)
profile_event_value_long = profile_event_value_long.drop(columns=['value_raw'])
profile_event_value_long['feature_key'] = (
    profile_event_value_long['source'].astype(str) + '__' + profile_event_value_long['feature'].astype(str)
)

# Категории ускоряют groupby и уменьшают память.
for col in ['profile_id', 'source', 'feature', 'feature_key', 'value_norm']:
    profile_event_value_long[col] = profile_event_value_long[col].astype('category')

print('profile_event_value_long shape:', profile_event_value_long.shape)


profile_event_value_long shape: (2110680, 9)


## 6. Словарь признаков

Обзор того, какие признаки получились после разворачивания списков и JSON.


In [10]:
feature_dictionary = (
    profile_event_value_long
    .groupby(['source', 'feature', 'feature_key'], observed=True, sort=False)
    .agg(
        token_count=('event_id', 'size'),
        n_profiles=('profile_id', 'nunique'),
        n_values=('value_norm', 'nunique'),
    )
    .reset_index()
)
# степень покрытия признака по профилям, а n_values — сколько разных значений встретилось у этого признака.
feature_dictionary['profile_share'] = feature_dictionary['n_profiles'] / raw['profile_id'].nunique()
feature_dictionary = feature_dictionary.sort_values(['source', 'n_profiles'], ascending=[True, False])
feature_dictionary.to_csv(OUTPUT_DIR / 'feature_dictionary.csv', index=False)

feature_dictionary.head(3)

,source,feature,feature_key,token_count,n_profiles,n_values,profile_share
14,fs,source_site_365,fs__source_site_365,84173,31684,335,0.511635
16,fs,has_click_365,fs__has_click_365,93619,26161,675,0.422449
17,fs,has_accept_365,fs__has_accept_365,163946,23784,664,0.384065


## 7. `profile_value_summary_long`: главный multi-value слой

Одна строка означает:

```text
у profile_id для feature_key встречалось value_norm N раз
```

Если у профиля было несколько значений одного признака, они остаются несколькими строками. Именно эта
таблица — источник данных для multi-value анализа.


In [11]:
profile_value_summary_long = (
    profile_event_value_long
    .groupby(['profile_id', 'source', 'feature', 'feature_key', 'value_norm'], observed=True, sort=False)
    .agg(
        value_count=('event_id', 'size'),
        first_seen=('created_at', 'min'),
        last_seen=('created_at', 'max'),
    )
    .reset_index()
)

first_raw = (
    profile_event_value_long
    .drop_duplicates(['profile_id', 'feature_key', 'value_norm'])
    [['profile_id', 'feature_key', 'value_norm', 'value_raw_text']]
    .rename(columns={'value_raw_text': 'value_raw_example'})
)

profile_value_summary_long = profile_value_summary_long.merge(
    first_raw,
    on=['profile_id', 'feature_key', 'value_norm'],
    how='left',
)

del first_raw
gc.collect()

# feature_observation_count
# Это общее число наблюдений конкретного признака у профиля.
feature_obs = (
    profile_value_summary_long
    .groupby(['profile_id', 'feature_key'], observed=True, sort=False)['value_count']
    .sum()
    .rename('feature_observation_count')
    .reset_index()
)

profile_value_summary_long = profile_value_summary_long.merge(
    feature_obs,
    on=['profile_id', 'feature_key'],
    how='left',
)

del feature_obs
gc.collect()
# value_share_within_feature
# Это доля конкретного значения внутри всех наблюдений
profile_value_summary_long['value_share_within_feature'] = (
    profile_value_summary_long['value_count'] / profile_value_summary_long['feature_observation_count']
)

profile_value_summary_long = profile_value_summary_long.sort_values(
    ['profile_id', 'feature_key', 'value_count', 'first_seen', 'value_norm'],
    ascending=[True, True, False, True, True],
)
# value_rank Ранг значения внутри признака у профиля.
# То есть value_rank = 1 — самое частое значение. Если частоты
# равны, берём более раннее, потом
# лексикографически меньшее для стабильности.
# Итого, это самое часто встречавшееся значение конкретного признака внутри конкретного профиля.
profile_value_summary_long['value_rank'] = (
    profile_value_summary_long
    .groupby(['profile_id', 'feature_key'], observed=True)
    .cumcount() + 1
)

# is_primary_value Флаг: is_primary_value = value_rank == 1 То есть это выбранное главное значение признака
# у профиля. Важно: это не замена остальных значений
profile_value_summary_long['is_primary_value'] = profile_value_summary_long['value_rank'].eq(1)

profile_value_summary_long.to_parquet(OUTPUT_DIR / 'profile_value_summary_long.parquet', index=False)

print('profile_value_summary_long shape:', profile_value_summary_long.shape)

profile_value_summary_long.head(2)

profile_value_summary_long shape: (1844644, 13)


,profile_id,source,feature,feature_key,value_norm,value_count,first_seen,last_seen,value_raw_example,feature_observation_count,value_share_within_feature,value_rank,is_primary_value
79672,000163e0-b9b4-3323-84ea-6495df7fe895,identity,email,identity__email,srnxztxqa8004@gmail.com,1,2026-05-09 07:47:48.739,2026-05-09 07:47:48.739,srnxztxqa8004@gmail.com,1,1.0,1,True
146579,000163e0-b9b4-3323-84ea-6495df7fe895,identity,sex,identity__sex,unknown,1,2026-05-09 07:47:48.739,2026-05-09 07:47:48.739,unknown,1,1.0,1,True


## 8. `blocking_index`: поведенческий и контекстный blocking

Здесь описаны основные элемены которые используются при блокинге.

`blocking_index` хранит many-to-many индекс:

```text
profile_id → block_family → block_rule → block_value
```

Главная идея: не строить blocking вокруг разреженных identity-полей. `email` почти уникален, `phone`, `last_name`, `birthday` заполнены слабо, поэтому они плохо подходят как основа группировки.

`block_family` — это тип или роль blocking-правила. Он нужен, чтобы не смешивать в одну кучу разные по смыслу правила.

В этом разделе используем семь семейств:

- `context` — контекстные ключи, в основном география: `rule__context__np_geoname_id`, `rule__context__np_subdivision`, `rule__context__rt_geoid`, `rule__context__rt_geoname`. Это основа для высокого охвата настоящих дублей, потому что гео-признаки заполнены существенно лучше identity-полей.
- `behavior` — поведенческие `fs` / site-id признаки: `rule__behavior__fs_source_site_365`, `rule__behavior__fs_source_site_30`, `rule__behavior__fs_visited_30`, `rule__behavior__fs_has_account` и другие. Они отражают поведенческий след профиля; совпадение по редкому site-id часто полезнее, чем совпадение по имени.
- `behavior_context` — композитные блоки вида география + поведенческий признак: например `rule__behavior_context__np_geoname_id__fs_source_site_365`, `rule__behavior_context__np_geoname_id__fs_has_click_365`. Такие блоки обычно чище: меньше случайных совпадений и меньше candidate pairs.
- `behavior_context_device` — более узкие вспомогательные правила для поиска дублей которые не поймали
основные правила, тюнинг. Например вида регион + поведенческий
признак + OS family.
- `cluster_candidate` — узкие кандидаты из кластерного анализа. Их не придумывали вручную: они взяты из разбора `cluster_only` пар, где кластеризация нашла реальные дубли, а текущие правила нет.
- `coverage_compound` — дополнительные композиты для профилей, которые не покрывались основной схемой: `geo + device/osfamily`, `email_domain + device/osfamily/sex`, `email_domain + email_initial2`. Это coverage-слой, но часть правил ещё и заметно повышает recall.
- `coverage_fallback` — технический fallback `rule__coverage_fallback__email_hash_bucket_1024`. Он гарантирует, что каждый профиль попадёт хотя бы в один блок размера `2..1000`. Это не сильный ER-сигнал, а защита от потери профилей на этапе candidate generation.
- `identity_rescue` — точечные телефонные правила: `rule__identity_rescue__phone_digits`, `rule__identity_rescue__phone_prefix6`. Это способ не потерять редкие, но сильные совпадения по телефону.

Итого, `rule__behavior__fs_has_account` — это именно правило blocking, а `fs__has_account` — это исходный признак в витрине.

Что происходит с остальными блоками:

- блоки размера `1` отбрасываются, потому что внутри них нет пары профилей для сравнения;
- блоки размера больше `1000` отбрасываются, потому что они создают слишком много шумных candidate pairs;
- такие значения не попадают в `blocking_index`, но сами признаки остаются в витрине и могут использоваться
 позже как similarity-признаки при сравнении уже найденных candidate pairs. А именно если для конкретного
 block_value число уникальных profile_id < 2 или > 1000, то этот block_value не используется в blocking_index

`block_family` — не фича профиля. Это метка самого blocking-правила, чтобы понимать, какую роль оно играет в генерации candidate pairs.

Чтобы ничего не потерять далее есть проверка, что правилам покрываются все профили.

Identity-признаки дальше лучше использовать как similarity/conflict features при сравнении candidate pairs.


In [12]:
pv = profile_value_summary_long

BLOCK_MIN_SIZE = 2
BLOCK_MAX_SIZE = 1000


def stable_hash_bucket(value, n_buckets: int) -> str:
    value = '' if pd.isna(value) else str(value)
    bucket = int(hashlib.md5(value.encode('utf-8')).hexdigest()[:8], 16) % n_buckets
    return str(bucket)


def derived_feature_values(feature: str, colname: str) -> pd.DataFrame:
    emails = pv[(pv['source'] == 'identity') & (pv['feature'] == 'email')][['profile_id', 'value_norm']].copy()
    emails['email_norm'] = emails['value_norm'].astype(str).str.lower()
    emails['email_domain'] = emails['email_norm'].str.extract(r'@([^@]+)$', expand=False).fillna('missing_domain')
    emails['email_local'] = emails['email_norm'].str.extract(r'^([^@]+)', expand=False).fillna('missing_local')
    emails['email_initial1'] = emails['email_local'].str[:1].replace('', 'missing')
    emails['email_initial2'] = emails['email_local'].str[:2].replace('', 'missing')
    emails['email_hash_bucket_1024'] = emails['email_local'].map(lambda value: stable_hash_bucket(value, 1024))

    if feature == 'registration_60m_bucket':
        # Приближение окна регистрации +/- 1 час через 30-минутные bucket-ключи.
        # Каждый профиль кладём в текущий bucket и два предыдущих. Тогда две регистрации
        # с разницей до 60 минут обязательно получат общий ключ, но часть пар до ~90 минут
        # тоже попадёт в кандидаты. Поэтому используем этот ключ только в композитах.
        times = profile_core[['profile_id', 'first_event_at']].dropna().copy()
        times['bucket_start'] = pd.to_datetime(times['first_event_at']).dt.floor('30min')
        parts = []
        for offset in [0, -1, -2]:
            tmp = times[['profile_id', 'bucket_start']].copy()
            tmp[colname] = (tmp['bucket_start'] + pd.to_timedelta(offset * 30, unit='m')).dt.strftime('%Y-%m-%d %H:%M')
            parts.append(tmp[['profile_id', colname]])
        return pd.concat(parts, ignore_index=True).drop_duplicates()

    if feature not in emails.columns:
        return pd.DataFrame(columns=['profile_id', colname])

    out = emails[['profile_id', feature]].rename(columns={feature: colname})
    return out[['profile_id', colname]].drop_duplicates()


def feature_values(source: str, feature: str, colname: str) -> pd.DataFrame:
    if source == 'derived':
        return derived_feature_values(feature, colname)

    sub = pv[(pv['source'] == source) & (pv['feature'] == feature)][['profile_id', 'value_norm']].copy()
    sub[colname] = sub['value_norm'].astype(str)
    return sub[['profile_id', colname]].drop_duplicates()

# Размер блока — это сколько уникальных profile_id имеют одно
# то же block_value внутри конкретного block_rule.
def apply_block_size_limits(
    sub: pd.DataFrame,
    block_value_col: str = 'block_value',
    min_block_size: int = BLOCK_MIN_SIZE,
    max_block_size: int = BLOCK_MAX_SIZE,
) -> pd.DataFrame:
    if sub.empty:
        return sub

    sizes = (
        sub.groupby(block_value_col, sort=False)['profile_id']
        .nunique()
        .rename('block_size')
        .reset_index()
    )
    keep_values = sizes[
        sizes['block_size'].between(min_block_size, max_block_size)
    ][[block_value_col]]

    return sub.merge(keep_values, on=block_value_col, how='inner')


def atomic_block(
    rule: str,
    source: str,
    feature: str,
    family: str,
    transform=None,
    min_len: int = 1,
    min_block_size: int = BLOCK_MIN_SIZE,
    max_block_size: int = BLOCK_MAX_SIZE,
) -> pd.DataFrame:
    """Создать blocking-правило по одному признаку.

    Пример: `rule__context__np_geoname_id` берёт все значения признака
    `source='np', feature='geoname_id'` из `profile_value_summary_long`.

    Для каждого профиля получается строка вида:
    `profile_id + block_family + block_rule + block_value`.

    Дальше применяются ограничения:
    - пустые значения удаляются;
    - слишком короткие значения удаляются через `min_len`;
    - блоки размера меньше `min_block_size` и больше `max_block_size` не попадают в индекс.

    Важно: `block_rule` — это имя правила, а не имя исходного признака.
    Поэтому используем префикс `rule__`, чтобы не путать правило с `feature_key`.
    """
    sub = feature_values(source, feature, 'block_value')
    if sub.empty:
        return pd.DataFrame(columns=['profile_id', 'block_family', 'block_rule', 'block_value'])

    if transform is not None:
        sub['block_value'] = sub['block_value'].map(transform)

    sub = sub[sub['block_value'].notna()]
    sub = sub[sub['block_value'].astype(str).str.len().ge(min_len)]
    sub = apply_block_size_limits(sub, 'block_value', min_block_size, max_block_size)
    sub['block_family'] = family
    sub['block_rule'] = rule

    return sub[['profile_id', 'block_family', 'block_rule', 'block_value']].drop_duplicates()


def composite_block(
    rule: str,
    specs: list[tuple],
    family: str,
    sep: str = '|',
    min_block_size: int = BLOCK_MIN_SIZE,
    max_block_size: int = BLOCK_MAX_SIZE,
) -> pd.DataFrame:
    """Создать blocking-правило по комбинации нескольких признаков.

    `specs` — список компонентов блока в формате:
    `(source, feature, output_colname, transform, min_len)`.

    Пример:
    `np.subdivision_1_iso_code + np.device + np.osfamily`.

    Как работает сборка:
    1. Для каждого компонента берём таблицу `profile_id + значение`.
    2. Удаляем пустые и слишком короткие значения.
    3. Склеиваем компоненты через inner join по `profile_id`.
       Значит, профиль попадёт в композитный блок только если у него есть все компоненты.
    4. Собираем `block_value` как строку из компонент через `sep`, например:
       `RU-MOW|smartphone|android`.
    5. Ограничиваем размер блока диапазоном `min_block_size..max_block_size`.
    6. Возвращаем строки `profile_id + block_family + block_rule + block_value`.

    Композит обычно точнее атомарного правила: совпадение только по региону слабое,
    а совпадение по региону + устройству + OS family уже заметно сужает область поиска.

    Например если у профиля есть:
        np.subdivision_1_iso_code = A, B
        np.device = mobile, desktop
        np.osfamily = android
        то composite_block() через inner join создаст комбинации:
        A|mobile|android
        A|desktop|android
        B|mobile|android
        B|desktop|android
        То есть профиль попадёт сразу в несколько composite block values.
        Это сделано в пользу recall: мы не теряем дубль только потому, что “главное” значение выбрали не то. Но минус — больше candidate pairs и больше шума.

    """
    # specs: (source, feature, output_colname, transform, min_len)
    out = None
    cols = []

    for source, feature, colname, transform, min_len in specs:
        vals = feature_values(source, feature, colname)

        if transform is not None:
            vals[colname] = vals[colname].map(transform)

        vals = vals[vals[colname].notna()]
        vals = vals[vals[colname].astype(str).str.len().ge(min_len)]

        out = vals if out is None else out.merge(vals, on='profile_id', how='inner')
        cols.append(colname)

    if out is None or out.empty:
        return pd.DataFrame(columns=['profile_id', 'block_family', 'block_rule', 'block_value'])

    out['block_value'] = out[cols].astype(str).agg(sep.join, axis=1)
    out = apply_block_size_limits(out[['profile_id', 'block_value']], 'block_value', min_block_size, max_block_size)
    out['block_family'] = family
    out['block_rule'] = rule

    return out[['profile_id', 'block_family', 'block_rule', 'block_value']].drop_duplicates()


prefix6 = lambda s: s[:6] if isinstance(s, str) and len(s) >= 6 else None


def daypart_bucket(value):
    try:
        hour = int(float(str(value)))
    except (TypeError, ValueError):
        return None
    if hour < 0 or hour > 23:
        return None
    if hour <= 5:
        return 'sleep_00_05'
    if hour <= 10:
        return 'morning_06_10'
    if hour <= 17:
        return 'work_11_17'
    return 'evening_18_23'


def weekend_bucket(value):
    try:
        day = int(float(str(value)))
    except (TypeError, ValueError):
        return None
    if day < 0 or day > 6:
        return None
    return 'weekend' if day in {5, 6} else 'weekday'


### Алгоритм создания первого слоя правил blocking

Первый слой правил нужен не для финального решения "это один клиент или нет", а для генерации разумного списка candidate pairs.
Идея: вместо сравнения всех профилей со всеми мы группируем профили по нескольким признакам и сравниваем только тех, кто попал в один блок.

### Общий алгоритм

1. Берём значение признака из `profile_value_summary_long`.

2. Нормализуем значение, если нужно:
   - приводим к строке;
   - убираем пустые значения;
   - для телефона можем оставить только цифры;
   - для email можем выделить домен, первые символы local-part или hash bucket.

3. Строим блок:
   - для атомарного правила: один признак -> один `block_value`;
   - для композитного правила: несколько признаков склеиваются в один `block_value`.

Пример атомарного правила:

```text
rule__context__np_geoname_id
profile_id = P1, np.geoname_id = 524901
-> block_value = 524901
```

Пример композитного правила:

```python
composite_block(
    'rule__coverage__np_subdivision__np_device__np_osfamily',
    [
        ('np', 'subdivision_1_iso_code', 'subdivision', None, 2),
        ('np', 'device', 'device', None, 1),
        ('np', 'osfamily', 'osfamily', None, 1),
    ],
    family='coverage_compound',
    max_block_size=1000,
)
```

Если у одного профиля несколько значений одного компонента, мы не выбираем "самое частое" значение. `feature_values()` отдаёт все уникальные значения профиля по этому признаку, а `composite_block()` делает merge компонентов. Поэтому профиль может попасть сразу в несколько композитных блоков. Это важно для recall: мы не теряем дубль только потому, что у профиля было несколько возможных device/osfamily/site-id значений.

Дальше сборка работает так:

1. Берём все значения `np.subdivision_1_iso_code` у профиля.
2. Берём все значения `np.device` у того же профиля.
3. Берём все значения `np.osfamily` у того же профиля.
4. Оставляем только комбинации, где есть все компоненты.
5. Склеиваем каждую комбинацию в ключ, например:

```text
subdivision=RU-MOW
device=smartphone
osfamily=android
-> block_value = RU-MOW|smartphone|android
```

6. Все профили с таким же `block_value` попадают в один блок.
7. Если в блоке меньше 2 профилей или больше 1000 профилей, блок не попадает в `blocking_index`.

То есть композитный блок не хранит отдельные признаки по отдельности. Он создаёт новый группировочный ключ из комбинации признаков.

### Как выбираем правила

Правила делятся на семейства.

| Семейство | Роль |
|---|---|
| `context` | Широкий слой охвата: география и контекст. Даёт процент настоящих дублей, но может шуметь. |
| `behavior` | Поведенческие `fs` / site-id признаки. Часто дают полезный сигнал, особенно если значение редкое. |
| `behavior_context` | Композиты вида география + поведенческий признак. Обычно чище, чем одиночный context или behavior. |
| `behavior_context_device` | Узкие композиты вида регион + поведение + устройство/OS. Дополнительный аккуратный слой. |
| `behavior_daypart` | Поведение + география + часть суток/будни-выходные. Добавлено как отдельная time-aware категория для подъёма recall. |
| `behavior_daypart_device` | Более узкий time-aware вариант: регион + поведение + OS + часть суток. |
| `registration_time_window` | Композиты с окном регистрации около часа. Время не используем отдельно: оно усиливает географию/устройство/поведение. |
| `identity_rescue` | Редкие, но сильные identity-совпадения, например телефон. Не основа blocking, а rescue-правила. |
| `coverage_compound` | Дополнительные композиты для профилей, которые плохо покрываются основными правилами. |
| `coverage_fallback` | Техническая страховка покрытия, а не сильный сигнал дубля. |



In [13]:
# Context rules: широкий recall-слой по географии/context.
# Они хорошо покрывают данные, но сами по себе слабые, поэтому дальше требуют evidence.
context_blocks = [
    atomic_block('rule__context__np_geoname_id', 'np', 'geoname_id', family='context', min_len=2, max_block_size=1000),
    atomic_block('rule__context__np_subdivision', 'np', 'subdivision_1_iso_code', family='context', min_len=2, max_block_size=1000),
    atomic_block('rule__context__rt_geoid', 'rt', 'geoid', family='context', min_len=2, max_block_size=1000),
    atomic_block('rule__context__rt_geoname', 'rt', 'geoname', family='context', min_len=2, max_block_size=1000),
]

# Behavior rules: поведенческие fs/site-id признаки.
# Их оставляем потому, что редкие site-id часто заметно сужают область поиска.
behavior_features = [
    'source_site_365',
    'source_site_30',
    'visited_30',
    'visited_365',
    'has_account',
    'has_click_365',
    'has_click_30',
    'has_accept_365',
    'has_accept_30',
    'has_order_365',
    'has_order_30',
    'has_view_90',
]

behavior_blocks = [
    atomic_block(
        f'rule__behavior__fs_{feature}',
        'fs',
        feature,
        family='behavior',
        min_len=1,
        max_block_size=1000,
    )
    for feature in behavior_features
]

# Behavior + context: география плюс поведение.
# Такие композиты обычно чище одиночных context/behavior правил.
behavior_context_features = [
    'source_site_365',
    'has_account',
    'has_click_365',
    'has_accept_365',
    'visited_30',
]

behavior_context_blocks = [
    composite_block(
        f'rule__behavior_context__np_geoname_id__fs_{feature}',
        [('np', 'geoname_id', 'geo', None, 2), ('fs', feature, 'fs_value', None, 1)],
        family='behavior_context',
        max_block_size=1000,
    )
    for feature in behavior_context_features
]

# Postman + context: коммуникационные признаки сами по себе шумные,
# поэтому добавляем только узкие композиты geo + postman.
# Идея эксперимента: XGBoost сможет использовать их как дополнительный
# weak evidence, не превращая postman в самостоятельную основу поиска.
postman_context_features = [
    'postman_action_90',
    'postman_campaign_90',
    'postman_response_90',
    'postman_action_30',
    'postman_campaign_30',
    'postman_response_30',
]

postman_context_blocks = [
    composite_block(
        f'rule__postman_context__np_geoname_id__fs_{feature}',
        [('np', 'geoname_id', 'geo', None, 2), ('fs', feature, 'fs_value', None, 1)],
        family='postman_context',
        max_block_size=1000,
    )
    for feature in postman_context_features
]
# Узкие behavior_context_device правила добавлены после rule discovery/deep-dive.
# Берём только несколько проверенных fs-признаков, чтобы не раздувать шум.
behavior_context_device_features = [
    'source_site_365',
    'has_click_365',
    'has_accept_365',
]

behavior_context_device_blocks = [
    composite_block(
        f'rule__behavior_context_device__np_subdivision__fs_{feature}__np_osfamily',
        [('np', 'subdivision_1_iso_code', 'subdivision', None, 2), ('fs', feature, 'fs_value', None, 1), ('np', 'osfamily', 'osfamily', None, 1)],
        family='behavior_context_device',
        max_block_size=1000,
    )
    for feature in behavior_context_device_features
]

# Behavior + daypart/weekend: time-aware category for recall lift.
# Keep only top 5 rules from the time blocking experiment: daypart + weekday/weekend.
behavior_daypart_features = [
    'source_site_365',
    'has_click_365',
    'has_accept_365',
]
behavior_daypart_blocks = [
    composite_block(
        f'rule__behavior_daypart__np_geoname_id__fs_{feature}__rt_daypart__rt_weekpart',
        [
            ('np', 'geoname_id', 'geo', None, 2),
            ('fs', feature, 'fs_value', None, 1),
            ('rt', 'local_hour', 'daypart', daypart_bucket, 8),
            ('rt', 'day', 'weekpart', weekend_bucket, 7),
        ],
        family='behavior_daypart',
        max_block_size=1000,
    )
    for feature in behavior_daypart_features
]

behavior_daypart_device_features = [
    'source_site_365',
    'has_click_365',
]
behavior_daypart_device_blocks = [
    composite_block(
        f'rule__behavior_daypart_device__np_subdivision__fs_{feature}__np_osfamily__rt_daypart__rt_weekpart',
        [
            ('np', 'subdivision_1_iso_code', 'subdivision', None, 2),
            ('fs', feature, 'fs_value', None, 1),
            ('np', 'osfamily', 'osfamily', None, 1),
            ('rt', 'local_hour', 'daypart', daypart_bucket, 8),
            ('rt', 'day', 'weekpart', weekend_bucket, 7),
        ],
        family='behavior_daypart_device',
        max_block_size=1000,
    )
    for feature in behavior_daypart_device_features
]

# Registration time window: это правило пришло напрямую из EDA, есть гипотеза, что повторы часто  регистрируются рядом по времени.
# Само время слишком шумное, поэтому добавляем его только как часть узких композитов.
# registration_60m_bucket — 30-минутные ключи с перекрытием, которые покрывают пары с разницей до часа.
registration_time_window_features = [
    'source_site_365',
    'has_click_365',
    'has_accept_365',
]

registration_time_window_blocks = [
    composite_block(
        'rule__registration_time_window__np_geoname_id__registration_60m',
        [
            ('np', 'geoname_id', 'geo', None, 2),
            ('derived', 'registration_60m_bucket', 'reg_60m', None, 16),
        ],
        family='registration_time_window',
        max_block_size=1000,
    ),
    composite_block(
        'rule__registration_time_window__np_subdivision__np_device__np_osfamily__registration_60m',
        [
            ('np', 'subdivision_1_iso_code', 'subdivision', None, 2),
            ('np', 'device', 'device', None, 1),
            ('np', 'osfamily', 'osfamily', None, 1),
            ('derived', 'registration_60m_bucket', 'reg_60m', None, 16),
        ],
        family='registration_time_window',
        max_block_size=1000,
    ),
] + [
    composite_block(
        f'rule__registration_time_window__np_geoname_id__fs_{feature}__registration_60m',
        [
            ('np', 'geoname_id', 'geo', None, 2),
            ('fs', feature, 'fs_value', None, 1),
            ('derived', 'registration_60m_bucket', 'reg_60m', None, 16),
        ],
        family='registration_time_window',
        max_block_size=1000,
    )
    for feature in registration_time_window_features
]

# Эти 3 узких кандидата взяты не вручную: их подсветил notebook 11
# на cluster_only парах, где кластеризация видела реальные дубли,
# а текущие рекомендованные правила их не ловили.
# Используем их только как evidence-кандидаты, не как самостоятельное решение о склейке.
cluster_candidate_blocks = [
    composite_block(
        'rule__cluster_candidate__np_is_not_russia__np_device__np_osfamily__np_browser',
        [
            ('np', 'is_not_russia', 'is_not_russia', None, 1),
            ('np', 'device', 'device', None, 1),
            ('np', 'osfamily', 'osfamily', None, 1),
            ('np', 'browser', 'browser', None, 1),
        ],
        family='cluster_candidate',
        max_block_size=1000,
    ),
    composite_block(
        'rule__cluster_candidate__fs_has_account__np_is_not_russia__np_device__np_osfamily__np_browser',
        [
            ('fs', 'has_account', 'fs_value', None, 1),
            ('np', 'is_not_russia', 'is_not_russia', None, 1),
            ('np', 'device', 'device', None, 1),
            ('np', 'osfamily', 'osfamily', None, 1),
            ('np', 'browser', 'browser', None, 1),
        ],
        family='cluster_candidate',
        max_block_size=1000,
    ),
    composite_block(
        'rule__cluster_candidate__fs_postman_campaign_90__fs_postman_action_90__fs_postman_response_90__np_device__np_osfamily',
        [
            ('fs', 'postman_campaign_90', 'campaign', None, 1),
            ('fs', 'postman_action_90', 'action', None, 1),
            ('fs', 'postman_response_90', 'response', None, 1),
            ('np', 'device', 'device', None, 1),
            ('np', 'osfamily', 'osfamily', None, 1),
        ],
        family='cluster_candidate',
        max_block_size=1000,
    ),
]

identity_rescue_blocks = [
    # Identity слишком разрежен для основы blocking. Оставляем только телефон как точечный high-precision ключ.
    atomic_block('rule__identity_rescue__phone_digits', 'identity', 'phone', family='identity_rescue', min_len=7, max_block_size=1000),
    atomic_block('rule__identity_rescue__phone_prefix6', 'identity', 'phone', family='identity_rescue', transform=prefix6, min_len=6, max_block_size=1000),
]

# Coverage compound: слабый слой для покрытия профилей, которые плохо ловятся основными правилами.
coverage_compound_blocks = [
    composite_block(
        'rule__coverage__np_geoname_id__np_device__np_osfamily',
        [('np', 'geoname_id', 'geo', None, 2), ('np', 'device', 'device', None, 1), ('np', 'osfamily', 'osfamily', None, 1)],
        family='coverage_compound',
        max_block_size=1000,
    ),
    composite_block(
        'rule__coverage__np_geoname_id__np_device__np_osfamily__np_browser',
        [('np', 'geoname_id', 'geo', None, 2), ('np', 'device', 'device', None, 1), ('np', 'osfamily', 'osfamily', None, 1), ('np', 'browser', 'browser', None, 1)],
        family='coverage_compound',
        max_block_size=1000,
    ),
    composite_block(
        'rule__coverage__np_subdivision__np_device__np_osfamily',
        [('np', 'subdivision_1_iso_code', 'subdivision', None, 2), ('np', 'device', 'device', None, 1), ('np', 'osfamily', 'osfamily', None, 1)],
        family='coverage_compound',
        max_block_size=1000,
    ),
    composite_block(
        'rule__coverage__email_domain__np_device__np_osfamily__sex',
        [('derived', 'email_domain', 'email_domain', None, 1), ('np', 'device', 'device', None, 1), ('np', 'osfamily', 'osfamily', None, 1), ('identity', 'sex', 'sex', None, 1)],
        family='coverage_compound',
        max_block_size=1000,
    ),
    composite_block(
        'rule__coverage__email_domain__email_initial2',
        [('derived', 'email_domain', 'email_domain', None, 1), ('derived', 'email_initial2', 'email_initial2', None, 1)],
        family='coverage_compound',
        max_block_size=1000,
    ),
]

coverage_fallback_blocks = [
    # Последний технический fallback для 100% покрытия профилей.
    atomic_block(
        'rule__coverage_fallback__email_hash_bucket_1024',
        'derived',
        'email_hash_bucket_1024',
        family='coverage_fallback',
        min_len=1,
        max_block_size=1000,
    ),
]

blocks = (
    context_blocks
    + behavior_blocks
    + behavior_context_blocks
    + postman_context_blocks
    + behavior_context_device_blocks
    + behavior_daypart_blocks
    + behavior_daypart_device_blocks
    + registration_time_window_blocks
    + cluster_candidate_blocks
    + identity_rescue_blocks
    + coverage_compound_blocks
    + coverage_fallback_blocks
)

blocking_index = (
    pd.concat([b for b in blocks if not b.empty], ignore_index=True)
    .drop_duplicates()
    .merge(profile_core[['profile_id', 'entity_id']], on='profile_id', how='left')
)

blocking_index.to_parquet(OUTPUT_DIR / 'blocking_index.parquet', index=False)

print('blocking_index shape:', blocking_index.shape)
print('blocking families:', blocking_index['block_family'].value_counts().to_dict())
blocking_index.head(2)


blocking_index shape: (1871187, 5)
blocking families: {'behavior_context': 351909, 'behavior': 350015, 'behavior_context_device': 225727, 'coverage_compound': 194295, 'behavior_daypart': 189050, 'context': 144941, 'registration_time_window': 124472, 'behavior_daypart_device': 101418, 'postman_context': 97189, 'coverage_fallback': 61927, 'cluster_candidate': 27562, 'identity_rescue': 2682}


,profile_id,block_family,block_rule,block_value,entity_id
0,000163e0-b9b4-3323-84ea-6495df7fe895,context,rule__context__np_geoname_id,625144,266eb686284a35db45a797ef75c01fc276e71fee1eb2aa...
1,00036d8f-9320-39b0-96e9-f0a457fc82a4,context,rule__context__np_geoname_id,538560,5b3e5b68690777e70e9f394d45ebb85aeb5f12c68b2f06...


## 9. Расчёт размера блоков

Технический расчёт размера блоков после отсечения слишком широких значений:

- сколько блоков даёт правило;
- какой максимальный размер блока;
- сколько candidate pairs получится внутри блоков;
- сколько профилей покрывает правило.

Эти показатели нужны дальше для отбора финального набора правил. Отдельный файл с ними не сохраняем: итоговая версия попадёт в `blocking_rule_quality_report`.



- `n_blocks` — сколько разных блоков создало правило. Например, если после фильтрации осталось 1118 разных `geoname_id`, то `n_blocks = 1118`.
- `max_block_size` — размер самого большого оставшегося `block_value`. `BLOCK_MAX_SIZE = 1000` — это верхняя граница для каждого отдельного `block_value`, а не лимит на `n_blocks`.
- `candidate_pairs` — сколько пар профилей создаёт правило внутри всех своих блоков. Для одного блока: `block_size * (block_size - 1) / 2`; затем суммируем по всем блокам.
- `n_profiles_covered` — сколько уникальных профилей попало хотя бы в один блок этого правила.
- `profile_coverage` — доля всех профилей, покрытых правилом: `n_profiles_covered / total_profiles`.


In [14]:
block_sizes = (
    blocking_index
    .groupby(['block_family', 'block_rule', 'block_value'], sort=False)
    .agg(
        block_size=('profile_id', 'nunique'),
        entity_nunique=('entity_id', 'nunique'),
    )
    .reset_index()
)
block_sizes['candidate_pairs'] = (block_sizes['block_size'] * (block_sizes['block_size'] - 1) // 2).astype('int64')

blocking_rule_size_report = (
    block_sizes
    .groupby(['block_family', 'block_rule'], sort=False)
    .agg(
        n_blocks=('block_value', 'size'),
        max_block_size=('block_size', 'max'),
        candidate_pairs=('candidate_pairs', 'sum'),
    )
    .reset_index()
)

rule_coverage = (
    blocking_index
    .groupby(['block_family', 'block_rule'])['profile_id']
    .nunique()
    .rename('n_profiles_covered')
    .reset_index()
)

n_profiles = raw['profile_id'].nunique()
total_possible_pairs = int(n_profiles * (n_profiles - 1) // 2)

blocking_rule_size_report = blocking_rule_size_report.merge(
    rule_coverage,
    on=['block_family', 'block_rule'],
    how='left',
)
blocking_rule_size_report['profile_coverage'] = blocking_rule_size_report['n_profiles_covered'] / n_profiles
blocking_rule_size_report = blocking_rule_size_report.sort_values(['block_family', 'candidate_pairs', 'max_block_size'])

blocking_rule_size_report.head(10)


,block_family,block_rule,n_blocks,max_block_size,candidate_pairs,n_profiles_covered,profile_coverage
14,behavior,rule__behavior__fs_has_order_30,83,751,352940,2034,0.032845
13,behavior,rule__behavior__fs_has_order_365,152,972,1189268,5629,0.090897
10,behavior,rule__behavior__fs_has_click_30,335,754,1820519,7769,0.125454
12,behavior,rule__behavior__fs_has_accept_30,220,831,1828901,7130,0.115136
7,behavior,rule__behavior__fs_visited_365,100,898,1915802,6005,0.096969
5,behavior,rule__behavior__fs_source_site_30,184,853,2557943,11796,0.190482
6,behavior,rule__behavior__fs_visited_30,283,990,3784535,10221,0.165049
9,behavior,rule__behavior__fs_has_click_365,570,744,4324067,14179,0.228963
4,behavior,rule__behavior__fs_source_site_365,272,963,6188234,16793,0.271174
8,behavior,rule__behavior__fs_has_account,638,974,14167865,14023,0.226444


## 10. Формирование финального набора правил

Цель раздела — получить конкретный список `block_rule`, который дальше уйдёт в анализ пар и модель.

Для каждого правила считаем:
- `positive_client_recall` — какую долю клиентов с повторами правило потенциально находит;
- `positive_pair_recall` — какую долю реальных пар дублей правило кладёт в кандидаты;
- `candidate_pairs` — нагрузка правила, то есть сколько пар оно создаёт;
- `max_block_size` — защита от слишком широких блоков;
- `recommended_for_next_step` — итоговый флаг: правило берём или нет.

Замечу! `entity_id` используется только здесь, для offline-оценки правил. В production-признаки он не
попадает.



In [15]:
from itertools import combinations

# Собираем все реальные пары дублей по разметке entity_id.
# Это нужно только для offline-оценки правил: в production entity_id недоступен.
positive_pair_rows = []
for entity_id, grp in profile_core.groupby('entity_id', sort=False):
    profiles = sorted(map(str, grp['profile_id'].dropna().unique()))
    if len(profiles) < 2:
        continue
    for left_profile_id, right_profile_id in combinations(profiles, 2):
        positive_pair_rows.append((len(positive_pair_rows), entity_id, left_profile_id, right_profile_id))

positive_pairs = pd.DataFrame(
    positive_pair_rows,
    columns=['pair_id', 'entity_id', 'profile_id_l', 'profile_id_r'],
)

# total_positive_pairs - знаменатель для positive_pair_recall.
total_positive_pairs = len(positive_pairs)
# repeat_entities - клиенты, у которых есть минимум два профиля.
# Это знаменатель для positive_client_recall.
repeat_entities = profile_core.loc[
    profile_core['entity_size'].fillna(0).astype(int).ge(2),
    'entity_id',
].dropna().drop_duplicates()
total_repeat_entities = int(repeat_entities.nunique())

def captured_positive_pairs_for_index(index_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if positive_pairs.empty or index_df.empty:
        return pd.DataFrame(columns=['block_family', 'block_rule', 'positive_pairs_captured'])

    # Для каждого правила проверяем, сколько реальных пар оно положило в один и тот же block_value.
    group_cols = ['block_family', 'block_rule']
    for (family, rule), rule_idx in index_df[['profile_id', 'block_family', 'block_rule', 'block_value']].drop_duplicates().groupby(group_cols, sort=False):
        left = positive_pairs.merge(
            rule_idx.rename(columns={'profile_id': 'profile_id_l'}),
            on='profile_id_l',
            how='inner',
        )
        captured = left.merge(
            rule_idx.rename(columns={'profile_id': 'profile_id_r'}),
            on=['profile_id_r', 'block_value'],
            how='inner',
            suffixes=('', '_r'),
        )
        rows.append((family, rule, int(captured['pair_id'].nunique())))

    return pd.DataFrame(rows, columns=['block_family', 'block_rule', 'positive_pairs_captured'])

def captured_repeat_entities_for_index(index_df: pd.DataFrame) -> pd.DataFrame:
    if index_df.empty:
        return pd.DataFrame(columns=['block_family', 'block_rule', 'repeat_entities_captured'])

    # Клиент считается пойманным правилом, если хотя бы два его профиля
    # оказались в одном block_value этого правила.
    captured = (
        index_df[['profile_id', 'entity_id', 'block_family', 'block_rule', 'block_value']]
        .dropna(subset=['entity_id'])
        .drop_duplicates(['block_family', 'block_rule', 'block_value', 'entity_id', 'profile_id'])
        .groupby(['block_family', 'block_rule', 'block_value', 'entity_id'], observed=True, sort=False)['profile_id']
        .nunique()
        .reset_index(name='profiles_from_entity_in_block')
    )
    captured = captured[captured['profiles_from_entity_in_block'].ge(2)]

    return (
        captured
        .groupby(['block_family', 'block_rule'], observed=True, sort=False)['entity_id']
        .nunique()
        .rename('repeat_entities_captured')
        .reset_index()
    )

# Считаем два основных показателя качества правила:
# 1) сколько реальных пар оно поймало;
# 2) сколько клиентов с повторами оно потенциально находит.
positive_pair_capture = captured_positive_pairs_for_index(blocking_index)
positive_entity_capture = captured_repeat_entities_for_index(blocking_index)

blocking_rule_quality_report = (
    blocking_rule_size_report
    .merge(positive_pair_capture, on=['block_family', 'block_rule'], how='left')
    .merge(positive_entity_capture, on=['block_family', 'block_rule'], how='left')
)
blocking_rule_quality_report['positive_pairs_captured'] = blocking_rule_quality_report['positive_pairs_captured'].fillna(0).astype('int64')
blocking_rule_quality_report['repeat_entities_captured'] = blocking_rule_quality_report['repeat_entities_captured'].fillna(0).astype('int64')
blocking_rule_quality_report['total_positive_pairs'] = total_positive_pairs
blocking_rule_quality_report['total_repeat_entities'] = total_repeat_entities

blocking_rule_quality_report['positive_pair_recall'] = np.where(
    total_positive_pairs > 0,
    blocking_rule_quality_report['positive_pairs_captured'] / total_positive_pairs,
    np.nan,
)
blocking_rule_quality_report['positive_client_recall'] = np.where(
    total_repeat_entities > 0,
    blocking_rule_quality_report['repeat_entities_captured'] / total_repeat_entities,
    np.nan,
)


blocking_rule_quality_report['pairs_per_positive_captured'] = np.where(
    blocking_rule_quality_report['positive_pairs_captured'] > 0,
    blocking_rule_quality_report['candidate_pairs'] / blocking_rule_quality_report['positive_pairs_captured'],
    np.inf,
)
FORCED_RECOMMENDED_RULES = {
    # Маленькое, но очень чистое rescue-правило: в текущем датасете 78/78 найденных пар правильные.
    'rule__identity_rescue__phone_digits',
}

#Настройка отбора правил

# Ограничения на число пар кандидатов, пока эмпирическое
MAX_RULE_CANDIDATE_PAIRS = 6_000_000
# Процент правил, по которому оно должно находить хотя бы указанную долю клиентов с повторами. Разумно было
# использовать, но очень много мелких блоков, не все ловлю
# если найдутся более интересные правила, то можно вернуть
# А так пока AssertionError: recommended rules profile coverage = 0.897799
#MIN_POSITIVE_CLIENT_RECALL = 0.005

# Базовый отбор: берём правила которые не должны создавать слишком много пар-кандидатов для дальнейшего
# сравнения и без слишком широких блоков.
# identity_rescue не берём массово: такие правила добавляются только явно через FORCED_RECOMMENDED_RULES.
base_recommended_mask = (
    #blocking_rule_quality_report['positive_client_recall'].ge(MIN_POSITIVE_CLIENT_RECALL) &
    blocking_rule_quality_report['candidate_pairs'].le(MAX_RULE_CANDIDATE_PAIRS)
    & blocking_rule_quality_report['max_block_size'].le(BLOCK_MAX_SIZE)
    & ~blocking_rule_quality_report['block_family'].eq('identity_rescue')
)
forced_recommended_mask = blocking_rule_quality_report['block_rule'].isin(FORCED_RECOMMENDED_RULES)

# Финальный флаг: именно эти правила дальше становятся свидетельствами совпадения по паре.
blocking_rule_quality_report['recommended_for_next_step'] = (
    base_recommended_mask | forced_recommended_mask
)
blocking_rule_quality_report['selection_reason'] = np.select(
    [forced_recommended_mask, base_recommended_mask],
    ['forced_rescue_rule', 'selected_by_size_and_load_limits'],
    default='not_selected',
)


blocking_rule_quality_report = blocking_rule_quality_report[[
    'block_family',
    'block_rule',
    'positive_client_recall',
    'repeat_entities_captured',
    'total_repeat_entities',
    'positive_pair_recall',
    'positive_pairs_captured',
    'total_positive_pairs',
    'candidate_pairs',
    'pairs_per_positive_captured',
    'max_block_size',
    'n_blocks',
    'n_profiles_covered',
    'profile_coverage',
    'selection_reason',
    'recommended_for_next_step',
]].sort_values(['positive_client_recall', 'candidate_pairs'], ascending=[False, True])

blocking_rule_quality_report.to_parquet(OUTPUT_DIR / 'blocking_rule_quality_report.parquet', index=False)
blocking_rule_quality_report.to_csv(OUTPUT_DIR / 'blocking_rule_quality_report.csv', index=False)

recommended_blocking_rules = blocking_rule_quality_report[
    blocking_rule_quality_report['recommended_for_next_step']
].copy()
recommended_blocking_rules.to_csv(OUTPUT_DIR / 'recommended_blocking_rules.csv', index=False)

# Минимальный guardrail: финальный набор правил должен покрывать все профили.
# Иначе часть профилей вообще не сможет породить пары-кандидаты на следующем шаге.
recommended_rule_keys = set(
    map(tuple, recommended_blocking_rules[['block_family', 'block_rule']].to_numpy())
)
recommended_index = blocking_index[
    pd.MultiIndex.from_frame(blocking_index[['block_family', 'block_rule']]).isin(recommended_rule_keys)
]
recommended_profile_coverage = recommended_index['profile_id'].nunique() / profile_core['profile_id'].nunique()
assert recommended_profile_coverage == 1.0, f'recommended rules profile coverage = {recommended_profile_coverage:.6f}'

print(f'recommended rules: {len(recommended_blocking_rules)}')
print(f'recommended profile coverage: {recommended_profile_coverage:.2%}')


recommended_blocking_rules[[
    'block_family',
    'block_rule',
    'positive_client_recall',
    'positive_pair_recall',
    'candidate_pairs',
    'pairs_per_positive_captured',
    'max_block_size',
    'selection_reason',
]].reset_index(drop=True)

recommended_blocking_rules.head(2)


recommended rules: 43
recommended profile coverage: 100.00%


,block_family,block_rule,positive_client_recall,repeat_entities_captured,total_repeat_entities,positive_pair_recall,positive_pairs_captured,total_positive_pairs,candidate_pairs,pairs_per_positive_captured,max_block_size,n_blocks,n_profiles_covered,profile_coverage,selection_reason,recommended_for_next_step
35,coverage_compound,rule__coverage__np_geoname_id__np_device__np_o...,0.539824,4358,8073,0.752613,12315,16363,5838365,474.085668,971,1831,44843,0.724127,selected_by_size_and_load_limits,True
29,context,rule__context__np_geoname_id,0.473182,3820,8073,0.719795,11778,16363,5688766,482.999321,980,1118,36865,0.595298,selected_by_size_and_load_limits,True


## 11. Быстрый статистический тест blocking-правила

Проверяем одно правило как статистическую гипотезу:

```text
rule__behavior_context_device__np_subdivision__fs_source_site_365__np_osfamily
```

**H0:** правило не лучше случайного сравнения профилей.

**H1:** среди пар, созданных правилом, доля настоящих дублей статистически выше случайной.


In [16]:
# Для примера берем не самое очевидное правило: без телефона/email,
# но с понятной логикой: регион + site-id + операционная система.
TARGET_RULE = 'rule__behavior_context_device__np_subdivision__fs_source_site_365__np_osfamily'

# Уровень значимости: если p-value ниже 1%, отвергаем H0.
ALPHA = 0.01

# Практический порог: правило должно быть не просто статистически значимым,
# а заметно лучше случайного перебора. Lift >= 50 означает минимум x50 к случайной базе.
MIN_LIFT = 50

# Контроль шума: если 95% блоков не больше 100 профилей,
# правило не создает массовые широкие группы кандидатов.
MAX_P95_BLOCK_SIZE = 100


def pairs_from_rule(index_df: pd.DataFrame) -> pd.DataFrame:
    # Один block_value = одна группа профилей с одинаковым ключом правила.
    # Внутри каждого такого блока правило предлагает сравнить все пары профилей.
    pair_rows = []
    rows = index_df[['block_value', 'profile_id']].drop_duplicates()

    for block_value, grp in rows.groupby('block_value', sort=False):
        profiles = sorted(grp['profile_id'].astype(str).unique())
        for i, left in enumerate(profiles[:-1]):
            for right in profiles[i + 1:]:
                pair_rows.append((left, right))

    # Одна и та же пара может встретиться повторно, если у профиля несколько значений.
    return pd.DataFrame(pair_rows, columns=['profile_id_l', 'profile_id_r']).drop_duplicates()


def count_all_positive_pairs(core: pd.DataFrame) -> int:
    # Считаем все реальные пары дублей по разметке entity_id.
    # Если у entity_id есть k профилей, то внутри него k * (k - 1) / 2 настоящих пар.
    sizes = core.groupby('entity_id', sort=False)['profile_id'].nunique()
    return int((sizes * (sizes - 1) // 2).sum())


# Берем из общего blocking_index только проверяемое правило.
rule_index = blocking_index[blocking_index['block_rule'].eq(TARGET_RULE)].copy()

# Строим все пары, которые это правило отправило бы дальше в модель.
rule_pairs = pairs_from_rule(rule_index)
profile_to_entity = profile_core.set_index('profile_id')['entity_id'].astype(str).to_dict()

# Разметка используется только для проверки качества правила.
# В production entity_id недоступен и в правило не входит.
rule_pairs['entity_id_l'] = rule_pairs['profile_id_l'].map(profile_to_entity)
rule_pairs['entity_id_r'] = rule_pairs['profile_id_r'].map(profile_to_entity)
rule_pairs['is_positive_pair'] = rule_pairs['entity_id_l'].eq(rule_pairs['entity_id_r'])

# observed_rate: фактическая доля настоящих дублей среди пар, созданных правилом.
n_pairs = len(rule_pairs)
positive_pairs = int(rule_pairs['is_positive_pair'].sum())
observed_rate = positive_pairs / n_pairs if n_pairs else 0.0

# random_rate: базовая вероятность случайно выбрать два профиля одного entity_id.
total_profiles = profile_core['profile_id'].nunique()
total_possible_pairs = total_profiles * (total_profiles - 1) // 2
total_positive_pairs = count_all_positive_pairs(profile_core)
random_rate = total_positive_pairs / total_possible_pairs

# Сколько настоящих дублей ожидали бы видеть в таком же числе пар при случайном выборе.
expected_random_positive_pairs = n_pairs * random_rate

# Lift показывает, во сколько раз правило лучше случайной базы.
lift = observed_rate / random_rate if random_rate else np.nan

# Проверяем, не достигается ли результат за счет слишком больших блоков.
block_sizes = rule_index.groupby('block_value', observed=True)['profile_id'].nunique()
p95_block_size = block_sizes.quantile(0.95) if len(block_sizes) else np.nan

# Биномиальный тест: могла ли такая или большая доля дублей возникнуть случайно.
p_value = binomtest(positive_pairs, n_pairs, random_rate, alternative='greater').pvalue

# Статистическая проверка: отвергаем H0, если p-value меньше ALPHA.
statistically_passed = p_value < ALPHA

# Практическая проверка: правило достаточно сильное и не создает слишком широкие блоки.
practically_passed = (lift >= MIN_LIFT) and (p95_block_size <= MAX_P95_BLOCK_SIZE)

# Итоговое решение требует обе проверки: статистика + практическая пригодность.
verdict = 'Используем' if statistically_passed and practically_passed else 'Не годится'

rule_hypothesis_test = pd.DataFrame([
    {
        'block_rule': TARGET_RULE,
        'verdict': verdict,
        'candidate_pairs': n_pairs,
        'positive_pairs': positive_pairs,
        'observed_positive_rate_pct': observed_rate * 100,
        'expected_random_positive_pairs': expected_random_positive_pairs,
        'lift_vs_random': lift,
        'p_value': p_value,
        'p95_block_size': p95_block_size,
    }
])

display(rule_hypothesis_test)


,block_rule,verdict,candidate_pairs,positive_pairs,observed_positive_rate_pct,expected_random_positive_pairs,lift_vs_random,p_value,p95_block_size
0,rule__behavior_context_device__np_subdivision_...,Используем,842224,3196,0.379471,7.187328,444.671506,0.0,27.0
